In [ ]:
# Install dependencies if needed (uncomment on first run)
# !pip install jax jaxlib dm-haiku optax matplotlib numpy

In [ ]:
import sys
sys.path.insert(0, '..')

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
import optax

# Import our modules
from src.data_gen import (
    generate_training_dataset,
    generate_slowness_map_2d,
    ricker_wavelet,
    sample_collocation_points_2d
)
from src.model import (
    WavePINN,
    WavePINNConfig,
    create_simple_pinn,
    predict_on_grid,
    predict_time_series
)

print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

## 1. Generate Training Data and Train Model

In [ ]:
# Generate synthetic data
print("Generating training data...")
data = generate_training_dataset(
    seed=42,
    nx=80,
    nz=80,
    n_interior=5000,
    n_boundary=1000,
    n_initial=500,
    base_velocity=2000.0,
    n_anomalies=4
)

print(f"Slowness map shape: {data['slowness'].shape}")
print(f"Velocity range: [{float(data['velocity'].min()):.0f}, {float(data['velocity'].max()):.0f}] m/s")
print(f"Interior points: {data['collocation']['interior'].shape[0]}")

In [ ]:
# Visualize the velocity model
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Velocity
im0 = axes[0].imshow(
    data['velocity'].T, 
    origin='lower', 
    cmap='viridis',
    extent=[0, 1, 0, 1]
)
axes[0].set_xlabel('x')
axes[0].set_ylabel('z')
axes[0].set_title('Velocity Model (m/s)')
plt.colorbar(im0, ax=axes[0])

# Slowness
im1 = axes[1].imshow(
    data['slowness'].T * 1000,  # Convert to ms/m for readability
    origin='lower',
    cmap='plasma',
    extent=[0, 1, 0, 1]
)
axes[1].set_xlabel('x')
axes[1].set_ylabel('z')
axes[1].set_title('Slowness Model (ms/m)')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Initialize and train the model
print("Initializing WavePINN model...")

config = WavePINNConfig(
    hidden_dims=[128, 128, 64],
    use_fourier_features=True,
    num_fourier_features=64,
    fourier_scale=10.0,
    lambda_pde=1.0,
    lambda_bc=10.0,
    lambda_ic=10.0
)

pinn = WavePINN(config, seed=42)
key = jr.PRNGKey(42)
params = pinn.init_params(key)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Model parameters: {n_params:,}")

In [ ]:
# Prepare training batch
batch = {
    'interior': data['collocation']['interior'],
    'boundary': data['collocation']['boundary'],
    'initial': data['collocation']['initial'],
    'velocity': data['velocity_at_interior']
}

# Training loop
print("Training model...")
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state, batch):
    def loss_fn(p):
        return pinn.total_loss(p, batch, return_components=False)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss

# Store training history
history = {'loss': [], 'pde': [], 'bc': [], 'ic': []}
n_epochs = 200

for epoch in range(n_epochs):
    params, opt_state, loss = train_step(params, opt_state, batch)
    
    if epoch % 20 == 0 or epoch == n_epochs - 1:
        _, components = pinn.total_loss(params, batch, return_components=True)
        history['loss'].append(float(loss))
        history['pde'].append(float(components['pde']))
        history['bc'].append(float(components['bc']))
        history['ic'].append(float(components['ic_u']))
        print(f"Epoch {epoch:4d}: Loss = {loss:.4e}, PDE = {components['pde']:.4e}")

print(f"\nTraining complete! Final loss: {history['loss'][-1]:.4e}")

## 2. Training Loss Curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = np.arange(0, n_epochs, 20).tolist() + [n_epochs - 1]

# Total loss
axes[0].semilogy(epochs[:len(history['loss'])], history['loss'], 'b-o', label='Total', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (Log Scale)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Component losses
axes[1].semilogy(epochs[:len(history['pde'])], history['pde'], 'r-o', label='PDE', linewidth=2)
axes[1].semilogy(epochs[:len(history['bc'])], history['bc'], 'g-s', label='BC', linewidth=2)
axes[1].semilogy(epochs[:len(history['ic'])], history['ic'], 'b-^', label='IC', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss Component')
axes[1].set_title('Loss Components')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Wavefield Snapshots

In [ ]:
# Predict wavefield at multiple time steps
time_steps = [0.05, 0.15, 0.25, 0.35, 0.45]
nx_plot, nz_plot = 100, 100

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

# Find global min/max for consistent colorscale
all_fields = []
for t in time_steps:
    u = predict_on_grid(pinn, params, nx_plot, nz_plot, t=t)
    all_fields.append(u)

vmax = max(float(jnp.max(jnp.abs(u))) for u in all_fields)
vmin = -vmax

for i, (ax, t, u) in enumerate(zip(axes, time_steps, all_fields)):
    im = ax.imshow(
        u.T,
        origin='lower',
        cmap='seismic',
        extent=[0, 1, 0, 1],
        vmin=vmin,
        vmax=vmax
    )
    ax.set_xlabel('x')
    if i == 0:
        ax.set_ylabel('z')
    ax.set_title(f't = {t:.2f}s')
    
    # Mark source location
    ax.plot(0.5, 0.1, 'k*', markersize=10)

plt.colorbar(im, ax=axes, label='Wavefield u(x,z,t)', shrink=0.8)
plt.suptitle('Wavefield Snapshots at Different Times', fontsize=14)
plt.tight_layout()
plt.show()

## 4. PDE Residual Heat-maps

In [ ]:
# Compute PDE residuals on a grid at fixed time
from src.data_gen import interpolate_velocity_at_coords

def compute_residual_grid(pinn, params, velocity_grid, t, nx=50, nz=50):
    """Compute PDE residual on a spatial grid at time t."""
    x = jnp.linspace(0.05, 0.95, nx)  # Avoid exact boundaries
    z = jnp.linspace(0.05, 0.95, nz)
    xx, zz = jnp.meshgrid(x, z, indexing='ij')
    
    coords = jnp.stack([
        xx.ravel(),
        zz.ravel(),
        jnp.full(nx * nz, t)
    ], axis=-1)
    
    # Interpolate velocity at grid points
    velocity = interpolate_velocity_at_coords(
        velocity_grid, coords[:, :2],
        x_range=(0.0, 1.0), z_range=(0.0, 1.0)
    )
    
    # Compute residual
    residual = pinn.compute_pde_residual_2d_efficient(params, coords, velocity)
    
    return residual.reshape(nx, nz), xx, zz

# Compute residuals at different times
times_residual = [0.1, 0.2, 0.3, 0.4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (ax, t) in enumerate(zip(axes, times_residual)):
    residual, xx, zz = compute_residual_grid(pinn, params, data['velocity'], t, nx=40, nz=40)
    
    # Plot log10(|residual|) for better visualization
    log_res = jnp.log10(jnp.abs(residual) + 1e-10)
    
    im = ax.pcolormesh(
        xx, zz, log_res,
        cmap='hot_r',
        shading='auto'
    )
    ax.set_xlabel('x')
    if i == 0:
        ax.set_ylabel('z')
    ax.set_title(f't = {t:.1f}s')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, label='log₁₀|residual|')

plt.suptitle('PDE Residual Heat-maps (lower = better)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Time Series at Receiver Locations

In [ ]:
# Predict time series at several receiver locations
receiver_locations = [
    (0.5, 0.5),   # Center
    (0.2, 0.8),   # Top-left
    (0.8, 0.8),   # Top-right
    (0.5, 0.9),   # Top-center
]

times = jnp.linspace(0, 0.5, 200)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (x, z) in zip(axes, receiver_locations):
    u_series = predict_time_series(pinn, params, location=(x, z), times=times)
    
    ax.plot(times, u_series, 'b-', linewidth=1.5)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Wavefield u')
    ax.set_title(f'Receiver at (x={x}, z={z})')
    ax.grid(True, alpha=0.3)

plt.suptitle('Wavefield Time Series at Receiver Locations', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Source Wavelet Visualization

In [ ]:
# Visualize the Ricker wavelet source
f0 = data['source_params']['f0']  # Dominant frequency
t = jnp.linspace(0, 0.2, 500)
wavelet = ricker_wavelet(t, f0=f0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time domain
axes[0].plot(t * 1000, wavelet, 'b-', linewidth=2)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Ricker Wavelet (f₀ = {f0} Hz)')
axes[0].grid(True, alpha=0.3)

# Frequency domain
dt = float(t[1] - t[0])
fft = jnp.fft.rfft(wavelet)
freqs = jnp.fft.rfftfreq(len(t), dt)

axes[1].plot(freqs, jnp.abs(fft), 'r-', linewidth=2)
axes[1].axvline(x=f0, color='blue', linestyle='--', label=f'f₀ = {f0} Hz')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Frequency Spectrum')
axes[1].set_xlim(0, 100)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Collocation Point Distribution

In [ ]:
# Visualize collocation point sampling
fig = plt.figure(figsize=(15, 5))

# Interior points (sample for visualization)
ax1 = fig.add_subplot(131, projection='3d')
interior = data['collocation']['interior'][:500]  # Sample
ax1.scatter(interior[:, 0], interior[:, 1], interior[:, 2], 
           c=interior[:, 2], cmap='viridis', s=2, alpha=0.5)
ax1.set_xlabel('x')
ax1.set_ylabel('z')
ax1.set_zlabel('t')
ax1.set_title('Interior Points (PDE)')

# Boundary points
ax2 = fig.add_subplot(132, projection='3d')
boundary = data['collocation']['boundary'][:200]
ax2.scatter(boundary[:, 0], boundary[:, 1], boundary[:, 2],
           c='red', s=5, alpha=0.5)
ax2.set_xlabel('x')
ax2.set_ylabel('z')
ax2.set_zlabel('t')
ax2.set_title('Boundary Points (BC)')

# Initial condition points
ax3 = fig.add_subplot(133)
initial = data['collocation']['initial']
ax3.scatter(initial[:, 0], initial[:, 1], c='green', s=10, alpha=0.5)
ax3.set_xlabel('x')
ax3.set_ylabel('z')
ax3.set_title('Initial Points (IC, t=0)')
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
# Print summary statistics
print("=" * 50)
print("WAVEPINN-NIF-SCALAR TRAINING SUMMARY")
print("=" * 50)

print(f"\nModel Architecture:")
print(f"  Hidden layers: {config.hidden_dims}")
print(f"  Fourier features: {config.num_fourier_features}")
print(f"  Total parameters: {n_params:,}")

print(f"\nTraining Data:")
print(f"  Grid size: {data['slowness'].shape}")
print(f"  Interior points: {data['collocation']['interior'].shape[0]}")
print(f"  Boundary points: {data['collocation']['boundary'].shape[0]}")
print(f"  Initial points: {data['collocation']['initial'].shape[0]}")

print(f"\nVelocity Model:")
print(f"  Min velocity: {float(data['velocity'].min()):.0f} m/s")
print(f"  Max velocity: {float(data['velocity'].max()):.0f} m/s")
print(f"  Mean velocity: {float(data['velocity'].mean()):.0f} m/s")

print(f"\nTraining Results:")
print(f"  Initial loss: {history['loss'][0]:.4e}")
print(f"  Final loss: {history['loss'][-1]:.4e}")
print(f"  Loss reduction: {(1 - history['loss'][-1]/history['loss'][0])*100:.1f}%")

# Final wavefield statistics
u_final = predict_on_grid(pinn, params, 50, 50, t=0.25)
print(f"\nWavefield at t=0.25s:")
print(f"  Max |u|: {float(jnp.max(jnp.abs(u_final))):.4f}")
print(f"  Mean u: {float(jnp.mean(u_final)):.6f}")
print(f"  Std u: {float(jnp.std(u_final)):.4f}")

print("\n" + "=" * 50)